In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

from src.load_data import load_data

save_path = "/Users/ppopov1/_archive/2026 OHBM (SPIs and holo)/adm-proj/data/pyspi_proc/"
families = ["cov_", "cov-sq_", "prec_"]

fbirn_data, demographics = load_data()
timeseries = fbirn_data["data"]
diagnoses = fbirn_data["diags"]
sexes = fbirn_data["sexes"]
ages = fbirn_data["ages"]

print(demographics)
print(f"# subjects: {timeseries.shape[0]}, # timepoints: {timeseries.shape[1]}, # features: {timeseries.shape[2]}")

In [ ]:
# === make matplotlib use Inter ===
import matplotlib as mpl
import matplotlib.font_manager as fm
from pathlib import Path

def setup_inter():
    # 1. Look for Inter font files in common macOS locations
    candidate_dirs = [
        Path.home() / "Library" / "Fonts",
        Path("/Library/Fonts"),
        Path("/System/Library/Fonts"),
        Path("/System/Library/Fonts/Supplemental"),
    ]
    inter_files = []
    for d in candidate_dirs:
        if d.exists():
            inter_files += list(d.glob("Inter*.ttf")) + list(d.glob("Inter*.otf"))

    # 2. Register any we find (matplotlib won't pick them up automatically
    #    if they were installed after the font cache was built)
    for f in inter_files:
        fm.fontManager.addfont(str(f))

    # 3. Check what Inter family names are now visible to matplotlib
    available = {f.name for f in fm.fontManager.ttflist}
    inter_names = sorted(n for n in available if n.lower().startswith("inter"))
    print(f"Inter font files found: {len(inter_files)}")
    print(f"Inter family names matplotlib sees: {inter_names}")

    # 4. Set Inter as the default sans-serif with sensible fallbacks
    family_name = inter_names[0] if inter_names else "Inter"
    mpl.rcParams["font.family"] = "sans-serif"
    mpl.rcParams["font.sans-serif"] = [family_name, "Helvetica", "Arial", "DejaVu Sans"]
    mpl.rcParams["pdf.fonttype"] = 42   # keep text as real text in SVG/PDF
    mpl.rcParams["svg.fonttype"] = "none"
    print(f"Default sans-serif → {mpl.rcParams['font.sans-serif']}")

setup_inter()

In [ ]:
ica_coords = pd.read_csv("data/ICN_coordinates.csv")
domains = ica_coords["Domain"]
# update nans with previous value
domains = domains.ffill()
domains = np.asarray(domains.tolist())

change_idx = np.flatnonzero(np.r_[True, domains[1:] != domains[:-1]])
# change_idx marks the start index of each group
starts = change_idx
# compute ends (inclusive) indices for each group
ends = np.r_[starts[1:] - 1, domains.size - 1]
centers = ((starts + ends) / 2.0).tolist()
# boundaries are positions between pixels: (end + 0.4) for each group except last
boundaries = (ends[:-1] + 0.4).tolist()

group_names_full = [domains[s] for s in starts]
group_names = ["SC", "AU", "SM", "VIS", "CC", "DM", "CB"]  # short names

In [ ]:
# load the data
import dill
with open(save_path+"good_data.pkl", 'rb') as f:
    DATA = dill.load(f) 
with open(save_path+"stats.pkl", 'rb') as f:
    STATS = dill.load(f)
with open(save_path+"proc_stats.pkl", 'rb') as f:
    PROC_STATS = dill.load(f)

In [ ]:
print(DATA.keys())

In [ ]:
PROC_STATS.keys()

In [ ]:
def ttest(data0, data1):
    stat, p_value = stats.ttest_ind(data0, data1, axis=0, equal_var=False)
    return stat, p_value

def analyze_group_differences(data, labels, stat_func = ttest, fdr=True, p_threshold=0.05):
    groups = np.unique(labels)

    C = data.shape[1]
    tril_indices = np.tril_indices(data.shape[1], k=-1)
    X = data[:, tril_indices[0], tril_indices[1]]
    n_samples = X.shape[0]
    X = X.reshape(n_samples, -1)

    group_data = [X[labels == g] for g in groups]

    stat, p_value = stat_func(group_data[0], group_data[1])
    if fdr:
        p_threshold = p_threshold/p_value.shape[0]
    p_thresh = (p_value < p_threshold).astype(int)

    # reshape importances back to matrix form
    full_stat, full_p_value, full_p_thresh = np.zeros((C, C)), np.zeros((C, C)), np.zeros((C, C))
    full_stat[tril_indices], full_p_value[tril_indices], full_p_thresh[tril_indices] = stat, p_value, p_thresh
    full_stat, full_p_value, full_p_thresh = full_stat + full_stat.T, full_p_value + full_p_value.T, full_p_thresh + full_p_thresh.T
    stat, p_value, p_thresh = full_stat, full_p_value, full_p_thresh

    # #######
    # group_data = [data[labels == g] for g in groups]

    # stat, p_value = stat_func(group_data[0], group_data[1])
    # if fdr:
    #     print(data.shape[1]*data.shape[2])
    #     p_threshold = p_threshold/data.shape[1]
    # p_thresh = (p_value < p_threshold).astype(int)
    # #######

    return stat, p_value, p_thresh.astype(bool)

def plot_heatmap(matrix, ax, cmap='bwr', vmin=None, vmax=None, guides_color='k'):
    cax = ax.imshow(matrix, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(centers)
    ax.set_xticklabels(group_names)
    ax.set_yticks(centers)
    ax.set_yticklabels(group_names)
    for boundary in boundaries:
        ax.axhline(boundary, color=guides_color, linewidth=1.5)
        ax.axvline(boundary, color=guides_color, linewidth=1.5)
    return cax

def plot_stats(stat, p_vals, p_thresh, title=""):
    fig, ax = plt.subplots(1, 3, figsize=(14, 5))
    cax1 = plot_heatmap(stat, ax[0], vmin=-5, vmax=5)
    ax[0].set_title("T-statistics" if title == "" else f"T-statistics ({title})")
    fig.colorbar(cax1, ax=ax[0], fraction=0.045)  

    cax2 = plot_heatmap(p_vals, ax[1], vmin=0, vmax=1, cmap='inferno_r')
    ax[1].set_title("p-values")
    fig.colorbar(cax2, ax=ax[1], fraction=0.045)  

    cax3 = plot_heatmap(p_thresh, ax[2], vmin=0, vmax=1, cmap='inferno', guides_color='white')
    ax[2].set_title("Significant p-values")
    cbar3 = fig.colorbar(cax3, ax=ax[2], fraction=0.045)
    cbar3.set_ticks([0, 1])
    cbar3.set_ticklabels(['False', 'True'])

    plt.tight_layout()
    plt.show()

def plot_means(data, labels, title="", unit_lim=True):
    groups = np.unique(labels)
    group_data = [data[diagnoses == g] for g in groups]
    means = [np.mean(gd, axis=0) for gd in group_data]

    if unit_lim:
        vmin, vmax = -1, 1
    else:
        vmax = max([np.max(np.abs(m)) for m in means])
        vmin = -vmax
        
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    cax = plot_heatmap(means[0], ax[0], cmap='bwr', vmin=vmin, vmax=vmax)
    fig.colorbar(cax, ax=ax[0], fraction=0.045)
    ax[0].set_title(f"Patients Mean {title}")
    cax = plot_heatmap(means[1], ax[1], cmap='bwr', vmin=vmin, vmax=vmax)
    fig.colorbar(cax, ax=ax[1], fraction=0.045)
    ax[1].set_title(f"Controls Mean {title}")
    plt.tight_layout()
    plt.show()

# pictures

In [ ]:
spis_to_plot = ['cov_GraphicalLassoCV', 'prec_GraphicalLassoCV', 'hsic', 'pec']
spis_titles = ['Covariance', 'Precision', 'HSIC', 'PEC']

unit_lim = False
fig, axes = plt.subplots(len(spis_to_plot), 3, figsize=(11, 3*len(spis_to_plot)))
for i, (spi, spi_title) in enumerate(zip(spis_to_plot, spis_titles)):
    ax = axes[i]
    data = DATA[spi]
    labels = DATA["diags"]
    groups = np.unique(labels)
    group_data = [data[labels == g] for g in groups]
    means = [np.mean(gd, axis=0) for gd in group_data]

    if unit_lim:
        vmin, vmax = -1, 1
    else:
        # print(means[0])
        vmax = max([np.nanmax(np.abs(m)) for m in means])
        vmin = -vmax
        
    cax = plot_heatmap(means[0], ax[0], cmap='bwr', vmin=vmin, vmax=vmax)
    # fig.colorbar(cax, ax=ax[0], fraction=0.045)
    fig.colorbar(cax, ax=ax[0], fraction=0.045)
    ax[0].set_title(f"SZ Mean {spi_title}")
    cax = plot_heatmap(means[1], ax[1], cmap='bwr', vmin=vmin, vmax=vmax)
    fig.colorbar(cax, ax=ax[1], fraction=0.045)
    ax[1].set_title(f"HC Mean {spi_title}")
    # plot_heatmap(means[0], ax[0], cmap='bwr', vmin=vmin, vmax=vmax)
    # ax[0].set_title(f"SZ Mean {spi_title}")
    # plot_heatmap(means[1], ax[1], cmap='bwr', vmin=vmin, vmax=vmax)
    # ax[1].set_title(f"HC Mean {spi_title}")


    stat = STATS[spi]["stat"]
    p_value = STATS[spi]["p_value"]
    p_thresh = STATS[spi]["p_thresh"]

    cax3 = plot_heatmap(p_thresh, ax[2], vmin=0, vmax=1, cmap='inferno', guides_color='white')
    ax[2].set_title(f"Significant p-values")
    cbar3 = fig.colorbar(cax3, ax=ax[2], fraction=0.045)
    cbar3.set_ticks([0, 1])
    cbar3.set_ticklabels(['F', 'T'])

plt.tight_layout()
plt.show()

In [ ]:
# === SPI gallery: one example matrix per SPI, 3x3 grid, saved as SVG ===
# Reuses plot_heatmap, centers, group_names, boundaries from earlier cells.

spi_panels = [
    # (key in DATA,                  display label,      colormap)
    ('cov_GraphicalLassoCV',         'Covariance',       'bwr'),
    ('prec_GraphicalLassoCV',        'Precision',        'bwr'),
    ('spearmanr',                    'Spearman ρ',       'bwr'),
    ('kendalltau',                   'Kendall τ',        'bwr'),
    ('xcorr_mean_sig-True',          'Mean cross-corr',  'bwr'),
    ('dcorr',                        'Distance corr',    'magma'),
    ('mgc',                          'MGC',              'magma'),
    ('hsic',                         'HSIC',             'magma'),
    ('pec',                          'Power env. corr',  'bwr'),
]

ncols = 3
nrows = int(np.ceil(len(spi_panels) / ncols))
panel_in = 3.3  # inches per panel

fig, axes = plt.subplots(nrows, ncols, figsize=(panel_in * ncols, panel_in * nrows))
axes = np.atleast_2d(axes)

for k, (key, label, cmap) in enumerate(spi_panels):
    ax = axes.flat[k]
    data = DATA[key]  # shape (n_subjects, 53, 53)

    # mean across all subjects, mask the diagonal so it doesn't dominate the scale
    M = np.nanmean(np.asarray(data, dtype=float), axis=0)
    np.fill_diagonal(M, np.nan)

    # robust per-panel limits (different SPIs live on wildly different scales)
    lo, hi = np.nanpercentile(M, [2, 98])
    if cmap in ('bwr', 'RdBu_r', 'seismic'):
        vmax = max(abs(lo), abs(hi))
        vmin = -vmax
        guides_color = 'k'
    else:
        vmin, vmax = max(lo, 0), hi
        guides_color = 'white'

    plot_heatmap(M, ax, cmap=cmap, vmin=vmin, vmax=vmax,
                 guides_color=guides_color)
    ax.set_title(label, fontsize=13, fontweight='bold')

# blank any unused cells
for k in range(len(spi_panels), nrows * ncols):
    axes.flat[k].axis('off')

plt.tight_layout()
plt.savefig('pics/spi_gallery_3x3.svg', format='svg', bbox_inches='tight')
plt.show()

In [ ]:
[print(x) for x in PROC_STATS.keys()]

In [ ]:
# === SVG export: all-SPI aggregation, permutation check, and per-SPI results ===
import os
import dill

os.makedirs('pics/results', exist_ok=True)


# ---------- helper: 3-panel triptych (works for aggregated & per-SPI) ----------
def plot_three_panel(stat_p, tree_p, title=None, aggregated=False,
                     save_path=None, show=True, figsize=(14, 5)):
    """
    aggregated=False -> bool inputs from a single SPI;
                        third panel = bitwise AND (stat_p & tree_p)
    aggregated=True  -> float [0,1] inputs already averaged across SPIs;
                        third panel = (stat_p + tree_p) / 2
    """
    fig, ax = plt.subplots(1, 3, figsize=figsize)

    if not aggregated:
        # Check if arrays contain fractional values (e.g. averaged across solvers)
        is_fractional = stat_p.dtype.kind in 'fc' and not np.array_equal(stat_p, stat_p.astype(bool))
        if is_fractional:
            and_matrix = (stat_p + tree_p) * 0.5
        else:
            and_matrix = stat_p.astype(bool) & tree_p.astype(bool)
        third_label = "AND Matrix"
        tick_pos, tick_labels = [0, 1], ['F', 'T']
        kw = {}                                  # let imshow auto-scale 0..1
    else:
        and_matrix = (stat_p + tree_p) * 0.5
        third_label = "Average of two"
        tick_pos, tick_labels = [0.0, 0.5, 1.0], [0.0, 0.5, 1.0]
        kw = {'vmin': 0, 'vmax': 1}

    panels = [(stat_p,     "Significant p"),
              (tree_p,     "RF importance"),
              (and_matrix, third_label)]

    for axx, (mat, lab) in zip(ax, panels):
        density = int(100 * np.sum(mat) / np.size(mat))
        cax = plot_heatmap(mat, axx, cmap='inferno',
                           guides_color='white', **kw)
        prefix = f"{title}: " if title else ""
        axx.set_title(f"{prefix}{lab} ({density}% density)")
        cbar = fig.colorbar(cax, ax=axx, fraction=0.045)
        cbar.set_ticks(tick_pos)
        cbar.set_ticklabels(tick_labels)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, format='svg', bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close(fig)


# ---------- 1. All-SPI aggregation (the main result) ----------
p_threshs, importance_masks = [], []
for spi in PROC_STATS:
    p_threshs.append(PROC_STATS[spi]["p_thresh"].astype(float))
    importance_masks.append(PROC_STATS[spi]["importance_mask"].astype(float))

agg_p  = np.mean(np.array(p_threshs),       axis=0)
agg_rf = np.mean(np.array(importance_masks), axis=0)

plot_three_panel(agg_p, agg_rf,
                 title="All SPIs", aggregated=True,
                 save_path='pics/results/result_all_spis.svg')


# ---------- 2. Permutation check (label-permuted null) ----------
perm_path = "data/permutations/"

perm_p_aggs, perm_rf_aggs = [], []
for i in range(20):
    with open(perm_path + f"perm_proc_stats_{i}.pkl", 'rb') as f:
        perm = dill.load(f)
    p_t = np.mean(np.array([perm[spi]["p_thresh"].astype(float)
                            for spi in perm]), axis=0)
    r_t = np.mean(np.array([perm[spi]["importance_mask"].astype(float)
                            for spi in perm]), axis=0)
    perm_p_aggs.append(p_t)
    perm_rf_aggs.append(r_t)

perm_p  = np.mean(np.array(perm_p_aggs),  axis=0)
perm_rf = np.mean(np.array(perm_rf_aggs), axis=0)

plot_three_panel(perm_p, perm_rf,
                 title="All SPIs (permuted)", aggregated=True,
                 save_path='pics/results/result_all_spis_perm.svg')


# ---------- 3. Per-SPI individual results ----------
# Default: produce one SVG for every SPI that has processed stats.
# If you only want a curated subset, set `spi_list = curated` below.
curated = [
    ('cov',  'Covariance'),
    ('prec', 'Precision'),
    ('spearmanr',             'Spearman'),
    ('kendalltau',            'Kendall tau'),
    ('xcorr_mean_sig-True',   'Mean cross-corr'),
    ('dcorr',                 'Distance corr'),
    ('mgc',                   'MGC'),
    ('hsic',                  'HSIC'),
    ('pec',                   'Power env. corr'),
]

# all SPIs available — uncomment to dump every one (37 figures)
# spi_list = [(k, k) for k in PROC_STATS.keys()]
spi_list = curated

for spi_key, spi_label in spi_list:
    if spi_key not in PROC_STATS:
        print(f"  ! {spi_key} not in PROC_STATS — skipping")
        continue
    stat_p = PROC_STATS[spi_key]["p_thresh"]
    tree_p = PROC_STATS[spi_key]["importance_mask"]
    safe   = spi_key.replace('/', '_').replace(' ', '_')
    plot_three_panel(stat_p, tree_p,
                     title=spi_label, aggregated=False,
                     save_path=f'pics/results/result_{safe}.svg',
                     show=False)
    print(f"  saved  result_{safe}.svg  ({spi_label})")

print("Done.")

In [ ]:
for k in ['cov', 'prec', 'kendalltau', 'dcorr', 'hsic']:
    p = np.asarray(PROC_STATS[k]['p_thresh'], dtype=float)
    print(f"{k:15s}  fractional={_is_fractional(p)}  "
          f"unique_vals={np.unique(p)[:5]}...")

In [ ]:
# === Combined results grid: per-SPI summary + aggregated bottom row ===
# Each SPI row: [mean connectivity matrix | per-SPI AND matrix]
# Bottom row:   [all-SPI mean AND          | permutation mean AND]
#
# Robust to family-aggregated SPIs (e.g. 'cov', 'prec') whose p_thresh /
# importance_mask are fractional rather than boolean — these use the
# "average of two" pattern, mirroring plot_three_panel(aggregated=True).

import os
import dill

os.makedirs('pics/results', exist_ok=True)


# ---------- helpers ----------

def _is_fractional(M):
    return np.any((M > 0) & (M < 1))


def get_spi_mean_matrix(key):
    """Mean connectivity matrix for an SPI key.

    If `key` is an exact DATA key, use it directly.
    Otherwise treat it as a family prefix and average DATA across all
    matching solver variants (e.g. 'cov' -> mean over cov_EmpiricalCovariance,
    cov_GraphicalLasso, ... — but not cov-sq_*).
    """
    if key in DATA:
        arr = np.asarray(DATA[key], dtype=float)
    else:
        matches = [k for k in DATA["spis"] if k.startswith(key + "_")]
        if not matches:
            raise KeyError(f"No DATA entries match {key!r}")
        # average per-subject matrices across solvers, keeping the subject axis
        arr = np.mean(
            np.array([np.asarray(DATA[k], dtype=float) for k in matches]),
            axis=0,
        )
    if arr.ndim == 3:
        return np.nanmean(arr, axis=0)
    if arr.ndim == 2:
        return arr.copy()
    raise ValueError(f"Unexpected shape {arr.shape} for SPI {key!r}")


def per_spi_and(p_thresh, importance_mask):
    """AND mask with automatic bool / fractional switching.

    bool inputs  (single SPI)            -> bitwise AND        -> {0, 1}
    fractional inputs (e.g. cov, prec)   -> (sp + tp) * 0.5    -> [0, 1]
    """
    sp = np.asarray(p_thresh,        dtype=float)
    tp = np.asarray(importance_mask, dtype=float)
    if _is_fractional(sp) or _is_fractional(tp):
        return (sp + tp) * 0.5
    return (sp.astype(bool) & tp.astype(bool)).astype(float)


def get_spi_and(key, stats=None):
    """Convenience wrapper around per_spi_and pulling from PROC_STATS (or override)."""
    stats = stats if stats is not None else PROC_STATS
    return per_spi_and(stats[key]["p_thresh"], stats[key]["importance_mask"])


# ---------- choose which SPIs to show ----------

rows = [
    # (key in DATA / PROC_STATS,   display label)
    ('cov',                        'Covariance'),         # aggregated across solvers
    ('prec',                       'Precision'),          # aggregated across solvers
    ('kendalltau',                 'Kendall τ'),
    ('xcorr_mean_sig-True',        'Mean cross-corr'),
    ('dcorr',                      'Distance corr'),
    ('hsic',                       'HSIC'),
    ('pec',                        'Power env. corr'),
]


# ---------- bottom-row aggregates ----------

# all-SPI mean AND across the full PROC_STATS bank
mean_and = np.mean(np.array([get_spi_and(k) for k in PROC_STATS]), axis=0)

# permutation mean AND (avg across 20 null runs of "avg across SPIs of AND")
perm_path = "data/permutations/"
perm_means = []
for i in range(20):
    with open(perm_path + f"perm_proc_stats_{i}.pkl", 'rb') as f:
        perm = dill.load(f)
    perm_means.append(
        np.mean(np.array([get_spi_and(k, perm) for k in perm]), axis=0)
    )
perm_mean_and = np.mean(np.array(perm_means), axis=0)


# ---------- plot ----------

nrows = len(rows) + 1
ncols = 2
panel_w, panel_h = 4.2, 3.6

fig, axes = plt.subplots(nrows, ncols,
                         figsize=(panel_w * ncols, panel_h * nrows))
axes = np.atleast_2d(axes)

# --- per-SPI rows ---
for r, (key, label) in enumerate(rows):
    # column 0 : mean connectivity matrix
    M = get_spi_mean_matrix(key)
    np.fill_diagonal(M, np.nan)
    lo, hi = np.nanpercentile(M, [2, 98])
    if lo < 0 and hi > 0:
        vmax = max(abs(lo), abs(hi)); vmin = -vmax
        cmap, guides = 'bwr', 'k'
    else:
        vmin, vmax = max(lo, 0), hi
        cmap, guides = 'magma', 'white'
    cax = plot_heatmap(M, axes[r, 0], cmap=cmap,
                       vmin=vmin, vmax=vmax, guides_color=guides)
    axes[r, 0].set_title(f"{label} · mean matrix")
    fig.colorbar(cax, ax=axes[r, 0], fraction=0.045)

    # column 1 : per-SPI AND matrix
    A = get_spi_and(key)
    cax = plot_heatmap(A, axes[r, 1], cmap='inferno',
                       vmin=0, vmax=1, guides_color='white')
    axes[r, 1].set_title(f"{label} · AND matrix")
    cbar = fig.colorbar(cax, ax=axes[r, 1], fraction=0.045)
    if _is_fractional(A):
        cbar.set_ticks([0.0, 0.5, 1.0]); cbar.set_ticklabels([0.0, 0.5, 1.0])
    else:
        cbar.set_ticks([0, 1]);           cbar.set_ticklabels(['F', 'T'])

# --- bottom row: all-SPI mean AND | permutation mean AND ---
for col, (mat, ttl) in enumerate([(mean_and,      "All SPIs · mean AND"),
                                  (perm_mean_and, "Permutation null · mean AND")]):
    cax = plot_heatmap(mat, axes[-1, col], cmap='inferno',
                       vmin=0, vmax=1, guides_color='white')
    axes[-1, col].set_title(ttl, fontweight='bold')
    cbar = fig.colorbar(cax, ax=axes[-1, col], fraction=0.045)
    cbar.set_ticks([0.0, 0.5, 1.0]); cbar.set_ticklabels([0.0, 0.5, 1.0])

plt.tight_layout()
plt.savefig('pics/results/results_grid_1.svg', format='svg', bbox_inches='tight')
plt.show()

In [ ]:
# === Replot all-SPI and permutation aggregate AND with each panel's own vmax ===
# (The structure in these matrices lives in roughly [0, 0.3], so the [0, 1]
#  scale we used in the full grid washes them out to dark.)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

for ax, mat, ttl in [(axes[0], mean_and,      "All SPIs · mean AND"),
                     (axes[1], perm_mean_and, "Permutation null · mean AND")]:
    vmax = float(np.nanmax(mat))
    cax  = plot_heatmap(mat, ax, cmap='inferno',
                        vmin=0, vmax=vmax, guides_color='white')
    ax.set_title(ttl, fontsize=13, fontweight='bold')
    cbar = fig.colorbar(cax, ax=ax, fraction=0.045)
    cbar.set_ticks([0, vmax / 2, vmax])
    cbar.set_ticklabels([f"{0:.2f}", f"{vmax/2:.2f}", f"{vmax:.2f}"])

plt.tight_layout()
plt.savefig('pics/results/results_aggregates.svg',
            format='svg', bbox_inches='tight')
plt.show()

In [ ]:
# === Replot all-SPI and permutation aggregate AND with each panel's own vmax ===
# (The structure in these matrices lives in roughly [0, 0.3], so the [0, 1]
#  scale we used in the full grid washes them out to dark.)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

for ax, mat, ttl in [(axes[0], mean_and,      "All SPIs · mean AND"),
                     (axes[1], perm_mean_and, "Permutation null · mean AND")]:
    vmax = float(np.nanmax(mat))
    cax  = plot_heatmap(mat, ax, cmap='inferno',
                        vmin=0, vmax=vmax, guides_color='white')
    ax.set_title(ttl, fontsize=13, fontweight='bold')
    cbar = fig.colorbar(cax, ax=ax, fraction=0.045)
    cbar.set_ticks([0, vmax / 2, vmax])
    cbar.set_ticklabels([f"{0:.2f}", f"{vmax/2:.2f}", f"{vmax:.2f}"])

plt.tight_layout()
plt.savefig('pics/results/results_aggregates.svg',
            format='svg', bbox_inches='tight')
plt.show()